In [1]:
import requests
import pandas as pd
from datetime import datetime
import time
import os
from google.cloud import storage
from google.cloud import bigquery

# --- CONFIGURATION ---
PROJECT_ID = "crypto-data-analysis-490804" # Aapka Project ID
BUCKET_NAME = "initial-layer-crypto-arbaz-2026" # Aapki nayi bucket
DATASET_ID = "initial_data"
TABLE_ID = "crypto_data_temp"
# ---------------------

def fetch_crypto_data():
    print("⏳ Fetching data from CoinGecko...")
    url = 'https://api.coingecko.com/api/v3/simple/price'

    current_date = datetime.now().strftime("%Y-%m-%d")
    current_date_filename = datetime.now().strftime("%Y%m%d")

    coins = ['bitcoin', 'ethereum', 'tether', 'binancecoin', 'solana']
    all_data = []

    for coin in coins:
        params = {
            'ids': coin,
            'vs_currencies': 'usd,inr,eur',
            'include_market_cap': 'true',
            'include_24hr_vol': 'true',
            'include_24hr_change': 'true',
            'include_last_updated_at': 'true',
        }

        try:
            response = requests.get(url, params=params)
            data = response.json()

            if coin in data:
                coin_data = data[coin]
                all_data.append({
                    'coin': coin,
                    'usd': coin_data.get('usd'),
                    'usd_market_cap': coin_data.get('usd_market_cap'),
                    'usd_24h_vol': coin_data.get('usd_24h_vol'),
                    'usd_24h_change': coin_data.get('usd_24h_change'),
                    'inr': coin_data.get('inr'),
                    'eur': coin_data.get('eur'),
                    'date': current_date
                })
        except Exception as e:
            print(f"❌ Error fetching {coin}: {e}")

        time.sleep(2) # API limit se bachne ke liye

    df = pd.DataFrame(all_data)
    file_path = f'crypto_data_{current_date_filename}.csv'
    df.to_csv(file_path, index=False)

    print(f"✅ Data saved locally: {file_path}")
    return file_path, current_date

def upload_to_gcs(file_path, current_date):
    print(f"⏳ Uploading {file_path} to GCS bucket: {BUCKET_NAME}...")
    try:
        client = storage.Client(project=PROJECT_ID)
        bucket = client.bucket(BUCKET_NAME)
        
        # GCS path: crypto_data/2026-03-20/crypto_data_20260320.csv
        blob_path = f'crypto_data/{current_date}/{file_path}'
        blob = bucket.blob(blob_path)
        
        blob.upload_from_filename(file_path)
        print(f"✅ Uploaded to GCS: gs://{BUCKET_NAME}/{blob_path}")
    except Exception as e:
        print(f"❌ GCS Upload Error: {e}")
        raise

def load_to_bigquery(current_date):
    print(f"⏳ Loading data from GCS to BigQuery ({DATASET_ID}.{TABLE_ID})...")
    try:
        client = bigquery.Client(project=PROJECT_ID)
        
        full_table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
        uri = f'gs://{BUCKET_NAME}/crypto_data/{current_date}/crypto_data_*.csv'

        job_config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,
            skip_leading_rows=1,
            autodetect=True, # Column names automatically detect kar lega
            write_disposition=bigquery.WriteDisposition.WRITE_APPEND, # Naya data purane mein add hoga
        )

        load_job = client.load_table_from_uri(uri, full_table_id, job_config=job_config)
        load_job.result() # Wait for job to finish

        print(f"✅ Successfully loaded to BigQuery table: {full_table_id}")
    except Exception as e:
        print(f"❌ BigQuery Load Error: {e}")
        raise

# --- EXECUTION ---
if __name__ == "__main__":
    try:
        # 1. Fetch
        local_file, date_str = fetch_crypto_data()
        
        # 2. Upload
        upload_to_gcs(local_file, date_str)
        
        # 3. Load
        load_to_bigquery(date_str)
        
        print("\n🎉 Pipeline completed successfully!")
    except Exception as main_err:
        print(f"\n⚠️ Pipeline failed: {main_err}")

⏳ Fetching data from CoinGecko...
✅ Data saved locally: crypto_data_20260320.csv
⏳ Uploading crypto_data_20260320.csv to GCS bucket: initial-layer-crypto-arbaz-2026...


C:\Users\Developer Arbaz\AppData\Roaming\Python\Python313\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ Uploaded to GCS: gs://initial-layer-crypto-arbaz-2026/crypto_data/2026-03-20/crypto_data_20260320.csv
⏳ Loading data from GCS to BigQuery (initial_data.crypto_data_temp)...
✅ Successfully loaded to BigQuery table: crypto-data-analysis-490804.initial_data.crypto_data_temp

🎉 Pipeline completed successfully!
